# CNN

Notebook ini dipakai untuk Bagian 3 dan Bagian 4 CNN: training 16 arsitektur Conv2D shared parameter, evaluasi macro F1-score, dan perbandingan Keras vs forward propagation from scratch.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "cnn":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/kennethpoenadi/Documents/Semester 6/ML/Tubes2')

In [2]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.run_cnn_experiments import config_from_yaml, generate_shared_conv_grid, run_grid
from tubes2_ml.cnn.evaluate import build_intel_test_dataset, evaluate_keras_model, evaluate_scratch_model
from tubes2_ml.scratch.models.cnn_classifier import build_scratch_cnn_from_keras
from tubes2_ml.visualization.feature_maps import save_conv_feature_visualizations
from tubes2_ml.visualization.grad_cam import save_gradcam

CONFIG_PATH = PROJECT_ROOT / "configs" / "cnn" / "shared_conv.yaml"
model_config, training_config = config_from_yaml(CONFIG_PATH)
model_config, training_config

(SharedConvCNNConfig(input_shape=(150, 150, 3), num_classes=6, conv_filters=(32, 64), kernel_sizes=(3, 3), pooling_type='max', dense_units=(128,), dropout_rate=0.3, learning_rate=0.001, activation='relu', name='shared_conv_cnn', compile_model=True, metrics=('accuracy',)),
 CNNTrainingConfig(train_dir=PosixPath('/Users/kennethpoenadi/Documents/Semester 6/ML/Tubes2/data/raw/intel_image_classification/seg_train/seg_train'), validation_dir=None, output_dir=PosixPath('/Users/kennethpoenadi/Documents/Semester 6/ML/Tubes2/models/keras/cnn'), history_dir=PosixPath('/Users/kennethpoenadi/Documents/Semester 6/ML/Tubes2/artifacts/experiments/cnn'), image_size=(150, 150), batch_size=32, epochs=10, validation_split=0.2, seed=42, save_format='keras'))

## Bagian 3: Pelatihan Model

Cell berikut menampilkan 16 variasi arsitektur: jumlah layer conv, kombinasi filter, kernel size, dan pooling.

In [3]:
grid = generate_shared_conv_grid(model_config)
print(f"Total experiments: {len(grid)}")

for index, config in enumerate(grid, start=1):
    print(
        f"{index:02d}. {config.name} | "
        f"filters={config.conv_filters} | kernels={config.kernel_sizes} | pooling={config.pooling_type}"
    )

Total experiments: 16
01. cnn_1conv_f16_k3_maxpool | filters=(16,) | kernels=(3,) | pooling=max
02. cnn_1conv_f16_k3_averagepool | filters=(16,) | kernels=(3,) | pooling=average
03. cnn_1conv_f16_k5_maxpool | filters=(16,) | kernels=(5,) | pooling=max
04. cnn_1conv_f16_k5_averagepool | filters=(16,) | kernels=(5,) | pooling=average
05. cnn_1conv_f32_k3_maxpool | filters=(32,) | kernels=(3,) | pooling=max
06. cnn_1conv_f32_k3_averagepool | filters=(32,) | kernels=(3,) | pooling=average
07. cnn_1conv_f32_k5_maxpool | filters=(32,) | kernels=(5,) | pooling=max
08. cnn_1conv_f32_k5_averagepool | filters=(32,) | kernels=(5,) | pooling=average
09. cnn_2conv_f16-32_k3-3_maxpool | filters=(16, 32) | kernels=(3, 3) | pooling=max
10. cnn_2conv_f16-32_k3-3_averagepool | filters=(16, 32) | kernels=(3, 3) | pooling=average
11. cnn_2conv_f16-32_k5-5_maxpool | filters=(16, 32) | kernels=(5, 5) | pooling=max
12. cnn_2conv_f16-32_k5-5_averagepool | filters=(16, 32) | kernels=(5, 5) | pooling=average
13

In [4]:
RUN_TRAINING = True

if RUN_TRAINING:
    run_grid(model_config, training_config)
else:
    print("Training skipped. Set RUN_TRAINING = True to train all 16 CNN models.")

Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 22s 62ms/step - accuracy: 0.3517 - loss: 1.5300 - val_accuracy: 0.5299 - val_loss: 1.3739
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 22s 62ms/step - accuracy: 0.4160 - loss: 1.3466 - val_accuracy: 0.6055 - val_loss: 1.2528
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 25s 71ms/step - accuracy: 0.4379 - loss: 1.2852 - val_accuracy: 0.6297 - val_loss: 1.1852
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 26s 74ms/step - accuracy: 0.4633 - loss: 1.2395 - val_accuracy: 0.6044 - val_loss: 1.1838
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 21s 59ms/step - accuracy: 0.4846 - loss: 1.2029 - val_accuracy: 0.5716 - val_loss: 1.1884
Epoch 6/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 19s 54ms/step - accuracy: 0.5071 - loss: 1.1670 - val_accuracy: 0.6033 - val_loss: 1.1307
Epoch 7/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 19s 55ms/step - accuracy: 0.520

2026-05-10 21:33:27.389229: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 18s 50ms/step - accuracy: 0.3318 - loss: 1.5852 - val_accuracy: 0.5791 - val_loss: 1.4330
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 18s 51ms/step - accuracy: 0.3884 - loss: 1.4444 - val_accuracy: 0.6269 - val_loss: 1.3389
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 18s 50ms/step - accuracy: 0.4188 - loss: 1.3787 - val_accuracy: 0.6297 - val_loss: 1.2758
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 17s 49ms/step - accuracy: 0.4613 - loss: 1.3005 - val_accuracy: 0.6326 - val_loss: 1.2252
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 17s 50ms/step - accuracy: 0.5058 - loss: 1.2266 - val_accuracy: 0.6404 - val_loss: 1.1636
Epoch 6/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 687s 2s/step - accuracy: 0.5316 - loss: 1.1673 - val_accuracy: 0.6311 - val_loss: 1.1403
Epoch 7/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 28s 81ms/step - accuracy: 0.5414

2026-05-10 21:59:44.715431: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 96ms/step - accuracy: 0.3401 - loss: 1.5463 - val_accuracy: 0.5182 - val_loss: 1.4634
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 183s 521ms/step - accuracy: 0.4172 - loss: 1.3508 - val_accuracy: 0.5249 - val_loss: 1.3033
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 43s 122ms/step - accuracy: 0.4659 - loss: 1.2468 - val_accuracy: 0.5695 - val_loss: 1.1814
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 165s 470ms/step - accuracy: 0.5039 - loss: 1.1737 - val_accuracy: 0.6262 - val_loss: 1.1086
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 34s 97ms/step - accuracy: 0.5311 - loss: 1.1271 - val_accuracy: 0.6736 - val_loss: 1.0402
Epoch 6/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 31s 89ms/step - accuracy: 0.5456 - loss: 1.1051 - val_accuracy: 0.7028 - val_loss: 1.0025
Epoch 7/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 32s 91ms/step - accuracy: 

2026-05-10 22:16:32.354374: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 35s 98ms/step - accuracy: 0.3631 - loss: 1.5004 - val_accuracy: 0.4907 - val_loss: 1.3954
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 37s 105ms/step - accuracy: 0.4312 - loss: 1.3178 - val_accuracy: 0.5887 - val_loss: 1.2572
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 38s 108ms/step - accuracy: 0.4707 - loss: 1.2380 - val_accuracy: 0.6069 - val_loss: 1.1700
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 40s 115ms/step - accuracy: 0.5134 - loss: 1.1750 - val_accuracy: 0.6589 - val_loss: 1.0641
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 40s 112ms/step - accuracy: 0.5371 - loss: 1.1314 - val_accuracy: 0.6764 - val_loss: 1.0387
Epoch 6/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 40s 114ms/step - accuracy: 0.5484 - loss: 1.1085 - val_accuracy: 0.6785 - val_loss: 1.0015
Epoch 7/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 46s 130ms/step - accuracy:

KeyboardInterrupt: 

## Ringkasan Hasil Bagian 3

Cell ini membaca metadata hasil training dari `artifacts/experiments/cnn`.

In [ ]:
history_dir = PROJECT_ROOT / training_config.history_dir
metadata_files = sorted(history_dir.glob("*.json"))

runs = []
for path in metadata_files:
    metadata = json.loads(path.read_text(encoding="utf-8"))
    cfg = metadata["model_config"]
    metrics = metadata.get("metrics", {})
    history = metadata.get("history", {})
    runs.append({"path": path, "config": cfg, "metrics": metrics, "history": history, "metadata": metadata})

if not runs:
    print(f"No training metadata found in {history_dir}.")
else:
    for run in sorted(runs, key=lambda item: item["metrics"].get("validation_macro_f1", -1), reverse=True):
        cfg = run["config"]
        print(cfg["name"], "| val_macro_f1=", run["metrics"].get("validation_macro_f1"))

No training metadata found in /Users/kennethpoenadi/Documents/Semester 6/ML/Tubes2/artifacts/experiments/cnn.


In [ ]:
if runs:
    plt.figure(figsize=(12, 6))
    for run in runs:
        history = run["history"]
        name = run["config"]["name"]
        if "val_loss" in history:
            plt.plot(history["val_loss"], label=name)
    plt.title("Validation Loss untuk 16 Arsitektur CNN")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.legend(fontsize=7, ncol=2)
    plt.show()

## Bagian 4: Eksperimen dan Evaluasi

Bagian ini memilih arsitektur terbaik dari Bagian 3 berdasarkan validation macro F1-score, lalu membandingkan Keras dan forward propagation scratch pada split test.

In [ ]:
if not runs:
    raise RuntimeError("Run Bagian 3 training first before Bagian 4 evaluation.")

best_run = max(runs, key=lambda item: item["metrics"].get("validation_macro_f1", -1))
best_metadata = best_run["metadata"]
best_name = best_run["config"]["name"]
model_path = PROJECT_ROOT / best_metadata["artifacts"]["model_path"]

print("Best model:", best_name)
print("Model path:", model_path)

keras_model = tf.keras.models.load_model(model_path)

RuntimeError: Run Bagian 3 training first before Bagian 4 evaluation.

In [ ]:
test_ds, class_names = build_intel_test_dataset(
    test_dir=PROJECT_ROOT / "data/raw/intel_image_classification/seg_test/seg_test",
    image_size=training_config.image_size,
    batch_size=training_config.batch_size,
)

keras_metrics = evaluate_keras_model(keras_model, test_ds, num_classes=len(class_names))
keras_metrics

In [ ]:
scratch_shared = build_scratch_cnn_from_keras(
    keras_model,
    replace_conv_with_local=False,
    input_shape=tuple(best_run["config"]["input_shape"]),
)

scratch_shared_metrics = evaluate_scratch_model(scratch_shared, test_ds, num_classes=len(class_names))
scratch_shared_metrics

In [ ]:
scratch_non_shared = build_scratch_cnn_from_keras(
    keras_model,
    replace_conv_with_local=True,
    input_shape=tuple(best_run["config"]["input_shape"]),
)

scratch_non_shared_metrics = evaluate_scratch_model(scratch_non_shared, test_ds, num_classes=len(class_names))

comparison = {
    "keras_shared_macro_f1": keras_metrics["macro_f1"],
    "scratch_shared_macro_f1": scratch_shared_metrics["macro_f1"],
    "scratch_non_shared_macro_f1": scratch_non_shared_metrics["macro_f1"],
    "keras_parameter_count": keras_model.count_params(),
    "scratch_shared_parameter_count": scratch_shared.count_parameters(),
    "scratch_non_shared_parameter_count": scratch_non_shared.count_parameters(),
}
comparison

## Bonus: Visualisasi Feature Maps dan Grad-CAM

Cell berikut menyimpan visualisasi intermediate feature maps dari layer Conv2D dan Grad-CAM untuk satu gambar test.

In [ ]:
sample_batch, sample_labels = next(iter(test_ds.take(1)))
sample_image = sample_batch[0].numpy()
sample_label = int(sample_labels[0].numpy())

plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.title(f"Ground truth: {class_names[sample_label]}")
plt.axis("off")
plt.show()

In [ ]:
feature_output_dir = PROJECT_ROOT / "artifacts/plots/cnn/feature_maps"
feature_paths = save_conv_feature_visualizations(
    model=keras_model,
    images=sample_image,
    output_dir=feature_output_dir,
    max_channels=16,
)

feature_paths

In [ ]:
predicted_class = int(tf.argmax(keras_model.predict(sample_image[None, ...], verbose=0)[0]).numpy())
gradcam_path = PROJECT_ROOT / "artifacts/plots/cnn/grad_cam" / f"{best_name}_sample_gradcam.png"

heatmap = save_gradcam(
    model=keras_model,
    image=sample_image,
    output_path=gradcam_path,
    class_index=predicted_class,
)

print("Predicted class:", class_names[predicted_class])
print("Grad-CAM saved to:", gradcam_path)
plt.imshow(heatmap, cmap="jet")
plt.axis("off")
plt.show()